In [ ]:
# Peak calling using downsample cells, 15,000 cells per cell type

In [ ]:
# Downsample Peak calling  500bp q=0.001

In [ ]:
# Downsample Peak calling  1000bp q=0.001 for chormbpnet

In [ ]:
import snapatac2 as snap
import numpy as np
import pandas as pd
import scanpy as sc
import polars as pl
import os 
snap.__version__

In [ ]:
myCustomGenome = snap.genome.Genome(fasta='/home/sunkeyong/softeware/ref_tree_shrew/TreeShrew_scATAC_ref/fasta/genome.fa', 
                                    annotation='/home/sunkeyong/softeware/ref_tree_shrew/TreeShrew_scATAC_ref/genes/genes.gtf')

In [ ]:
os.chdir('snapATAC2_downsample/Peak_calling_each_celltype')

In [ ]:
######### 循环解决
# 定义所有细胞类型
cell_types = ["C1.1","C10.1","C11.1","C11.2","C11.3","C11.4","C12.1","C12.10","C12.11","C12.12","C12.2","C12.3","C12.4",
              "C12.5","C12.6","C12.7","C12.8","C12.9","C13.1","C14.1","C14.10","C14.11","C14.12","C14.13","C14.14","C14.15","C14.16","C14.17",
              "C14.18","C14.19","C14.2","C14.20","C14.21","C14.22","C14.23","C14.24","C14.3","C14.4","C14.5","C14.6","C14.7","C14.8","C14.9",
              "C15.1","C15.2","C16.1","C16.2","C16.3","C16.4","C16.5","C16.6","C16.7","C17.1","C17.2","C17.3","C17.4","C18.1","C18.10","C18.11","C18.12","C18.13",
              "C18.14","C18.15","C18.16","C18.17","C18.18","C18.19","C18.2","C18.20","C18.3","C18.4","C18.5","C18.6","C18.7","C18.8","C18.9",
              "C19.1","C2.1","C2.2","C2.3","C2.4","C2.5","C2.6","C2.7","C20.1","C21.1","C22.1","C23.1","C23.10","C23.11","C23.12",
              "C23.13","C23.2","C23.3","C23.4","C23.5","C23.6","C23.7","C23.8","C23.9","C24.1","C24.10","C24.11","C24.12",
              "C24.13","C24.14","C24.15","C24.16","C24.17","C24.18","C24.19","C24.2","C24.20","C24.21","C24.22","C24.23","C24.3",
              "C24.4","C24.5","C24.6","C24.7","C24.8","C24.9","C3.1","C3.2","C4.1","C5.1","C5.10","C5.11","C5.12","C5.13","C5.14","C5.15",
              "C5.16","C5.17","C5.18","C5.19","C5.2","C5.20","C5.21","C5.22","C5.23","C5.24","C5.25","C5.26","C5.27","C5.3","C5.4","C5.5",
              "C5.6","C5.7","C5.8","C5.9","C6.1","C6.2","C7.1","C7.10","C7.2","C7.3","C7.4","C7.5","C7.6","C7.7","C7.8","C7.9","C8.1","C8.10","C8.11","C8.12","C8.13",
              "C8.14","C8.15","C8.16","C8.17","C8.18","C8.19","C8.2","C8.20","C8.21","C8.22","C8.23","C8.24","C8.25","C8.26","C8.27","C8.28",
              "C8.3","C8.4","C8.5","C8.6","C8.7","C8.8","C8.9","C9.1","C9.2","C9.3","C9.4"]  

In [ ]:
# 循环处理每个细胞类型
for cell_type in cell_types:
    print(f"正在处理细胞类型: {cell_type}")
    
    try:
        # 1. 准备数据路径
        h5ad_file = f'../Peak_calling/{cell_type}.h5ad'
        
        # 2. 创建AnnDataSet
        xdata = snap.AnnDataSet(
            adatas=[(cell_type, h5ad_file)],
            filename=f'{cell_type}.h5ads'  # 临时文件
        )
        
        # 3. 运行MACS3
        snap.tl.macs3(xdata, groupby='sample', shift=-75, extsize=150, n_jobs=3, qvalue=0.001)
        
        # 4. 合并peaks
        peaks = snap.tl.merge_peaks(xdata.uns['macs3'], myCustomGenome, half_width=250)
        
        # 5. 转换为BED格式
        if 'Peaks' in peaks and len(peaks['Peaks']) > 0:
            peaks_tmp = peaks['Peaks']
            df = pd.DataFrame(peaks_tmp, columns=['peak'])
            df[['chrom', 'positions']] = df['peak'].str.split(':', expand=True)
            df[['start', 'end']] = df['positions'].str.split('-', expand=True)
            df['start'] = df['start'].astype(int)
            df['end'] = df['end'].astype(int)
            
            # 创建BED文件
            bed_df = df[['chrom', 'start', 'end']]
            output_file = f'./500bp_peak_list/{cell_type}.bed'
            bed_df.to_csv(output_file, sep='\t', header=False, index=False)
            
            print(f"  ✓ 成功生成 {len(bed_df)} 个peaks到 {output_file}")
        else:
            print(f"  ✗ 未找到peaks: {cell_type}")

        # 4. 合并peaks
        peaks = snap.tl.merge_peaks(xdata.uns['macs3'], myCustomGenome, half_width=500)
        
        # 5. 转换为BED格式
        if 'Peaks' in peaks and len(peaks['Peaks']) > 0:
            peaks_tmp = peaks['Peaks']
            df = pd.DataFrame(peaks_tmp, columns=['peak'])
            df[['chrom', 'positions']] = df['peak'].str.split(':', expand=True)
            df[['start', 'end']] = df['positions'].str.split('-', expand=True)
            df['start'] = df['start'].astype(int)
            df['end'] = df['end'].astype(int)
            
            # 创建BED文件
            bed_df = df[['chrom', 'start', 'end']]
            output_file = f'./1000bp_peak_list/{cell_type}.bed'
            bed_df.to_csv(output_file, sep='\t', header=False, index=False)
            
            print(f"  ✓ 成功生成 {len(bed_df)} 个peaks到 {output_file}")
        else:
            print(f"  ✗ 未找到peaks: {cell_type}")
     
        # 6. 关闭文件
        xdata.close()
        
    except FileNotFoundError as e:
        print(f"  ✗ 文件未找到: {h5ad_file}")
    except Exception as e:
        print(f"  ✗ 处理{cell_type}时出错: {str(e)}")
    print("-" * 50)
print("所有细胞类型处理完成！")